# Notebook 09 — Selectivity Analysis & Final Candidate Ranking

This notebook integrates all experimental results into publication-quality figures
and a final candidate shortlist.

| Section | Analysis | Key output |
|---------|----------|-----------|
| 1 | Polished WT-vs-Mutant scatter (3 cohorts) | Main figure |
| 2 | Statistical tests (KS + Mann-Whitney) | p-values |
| 3 | Tanimoto structural selectivity score | Secondary selectivity signal |
| 4 | Top-10 candidates composite ranking | Candidate shortlist |
| 5 | 3D case study — best T790M-selective candidate | Structural insight |
| 6 | Findings summary | Report table |

**Runtime:** CPU only.


In [ ]:
# -- Bootstrap --
import os, sys, subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = '/content/drive/MyDrive/egfr_diffusion'
except ImportError:
    PROJECT_ROOT = os.path.abspath('.')

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
subprocess.run(['git', '-C', PROJECT_ROOT, 'pull'], check=False)

for pkg in ['rdkit', 'pandas', 'matplotlib', 'scipy', 'py3Dmol']:
    try:
        __import__(pkg if pkg != 'py3Dmol' else 'py3Dmol')
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=False)

import rdkit, pandas as pd, numpy as np, matplotlib.pyplot as plt
print('PROJECT_ROOT:', PROJECT_ROOT)
print('rdkit:', rdkit.__version__)

# Load all data once
SCORES    = f'{PROJECT_ROOT}/results/vina_scores/combined_scores.csv'
SCORES_6A = f'{PROJECT_ROOT}/results/vina_scores/4I22_6A_cross.csv'
FILTERED_WT  = f'{PROJECT_ROOT}/results/filtered/1M17_filtered.csv'
FILTERED_MUT = f'{PROJECT_ROOT}/results/filtered/4I22_filtered.csv'
BASELINES    = f'{PROJECT_ROOT}/results/filtered/baselines.csv'

df       = pd.read_csv(SCORES)
df6a     = pd.read_csv(SCORES_6A) if os.path.exists(SCORES_6A) else pd.DataFrame()
filt_wt  = pd.read_csv(FILTERED_WT)
filt_mut = pd.read_csv(FILTERED_MUT)
baselines_props = pd.read_csv(BASELINES)

df['delta']  = df['vina_Mut'] - df['vina_WT']
if not df6a.empty:
    df6a = df6a[(df6a['status_WT']=='ok') & (df6a['status_Mut']=='ok')].copy()
    df6a['delta'] = df6a['vina_Mut'] - df6a['vina_WT']
    df6a['source'] = 'Mut_6A'

gen   = df[df['source'] != 'baseline'].copy()
bases = df[df['source'] == 'baseline'].copy()

import yaml
cfg = yaml.safe_load(open(f'{PROJECT_ROOT}/configs/targets.yaml'))
print(f'Loaded {len(df)} cross-docked mols, {len(df6a)} 6A mols')


---
## Section 1 — Polished WT vs T790M Selectivity Scatter

Three cohorts on one figure:
- **Blue:** WT-conditioned (10 A), n=50
- **Red:** T790M-conditioned (10 A), n=50
- **Orange:** T790M-conditioned (6 A), n=50 (Experiment A)
- **Green stars:** Erlotinib / Gefitinib / Osimertinib

Quadrant labels reflect clinical selectivity expectations.


In [ ]:
# -- Section 1: Polished scatter --
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(8, 7))

# ---- generated cohorts ----
cohorts = [
    ('WT_generated',  '#1f77b4', 'o', 50, 'WT-cond 10A (n=50)'),
    ('Mut_generated', '#d62728', 's', 50, 'Mut-cond 10A (n=50)'),
]
for src, col, mk, sz, lbl in cohorts:
    sub = gen[gen['source'] == src]
    ax.scatter(sub['vina_WT'], sub['vina_Mut'],
               c=col, marker=mk, s=sz, alpha=0.55,
               edgecolors='none', label=lbl)

if not df6a.empty:
    ax.scatter(df6a['vina_WT'], df6a['vina_Mut'],
               c='#e67e22', marker='^', s=40, alpha=0.55,
               edgecolors='none', label='Mut-cond 6A (n=50)')

# ---- baselines ----
baseline_styles = {
    'erlotinib':   ('*', '#2ca02c', 220, 'Erlotinib (WT)'),
    'gefitinib':   ('*', '#9467bd', 220, 'Gefitinib (WT)'),
    'osimertinib': ('*', '#ff7f0e', 220, 'Osimertinib (T790M)'),
}
for _, row in bases.iterrows():
    name = row.get('name', row['source']).lower()
    for key, (mk, col, sz, lbl) in baseline_styles.items():
        if key in name:
            ax.scatter(row['vina_WT'], row['vina_Mut'],
                       marker=mk, c=col, s=sz, zorder=5,
                       edgecolors='black', linewidths=0.8, label=lbl)
            ax.annotate(lbl,
                        (row['vina_WT'], row['vina_Mut']),
                        xytext=(7, -14), textcoords='offset points',
                        fontsize=8.5, color=col, fontweight='bold')

# ---- diagonal (equal binding) ----
all_x = list(gen['vina_WT']) + list(bases['vina_WT'])
all_y = list(gen['vina_Mut']) + list(bases['vina_Mut'])
if not df6a.empty:
    all_x += list(df6a['vina_WT']); all_y += list(df6a['vina_Mut'])
lo = min(min(all_x), min(all_y)) - 0.5
hi = max(max(all_x), max(all_y)) + 0.5
ax.plot([lo, hi], [lo, hi], '--', color='gray', alpha=0.45,
        linewidth=1.2, label='Equal binding')

# ---- quadrant lines at median ----
med = np.median(list(gen['vina_WT']) + list(gen['vina_Mut']))
ax.axvline(med, color='#95a5a6', linewidth=0.7, alpha=0.5)
ax.axhline(med, color='#95a5a6', linewidth=0.7, alpha=0.5)

# ---- quadrant labels ----
q_kw = dict(fontsize=8, alpha=0.55, style='italic',
            bbox=dict(facecolor='white', alpha=0.4, edgecolor='none', pad=2))
ax.text(lo+0.3, hi-0.3, 'T790M-selective (Mut better)', color='#d62728', **q_kw)
ax.text(hi-2.5, lo+0.3, 'WT-selective (WT better)', color='#1f77b4', **q_kw)
ax.text(hi-2.5, hi-0.3, 'Dual binder',                 color='gray',    **q_kw)

ax.invert_xaxis(); ax.invert_yaxis()
ax.set_xlabel('Vina score on WT (1M17), kcal/mol  (lower = better)', fontsize=11)
ax.set_ylabel('Vina score on T790M (4I22), kcal/mol  (lower = better)', fontsize=11)
ax.set_title('Cross-docking Selectivity: WT vs T790M  (delta=vina_Mut-vina_WT; delta<0: T790M-selective)', fontsize=11)
ax.legend(fontsize=8.5, loc='lower right', framealpha=0.85)
ax.grid(alpha=0.25)

plt.tight_layout()
out = f'{PROJECT_ROOT}/results/analysis/selectivity_scatter_final.png'
plt.savefig(out, dpi=160, bbox_inches='tight')
plt.show()
print('Saved ->', out)


---
## Section 2 — Statistical Tests

Three comparisons using two-sample tests:
1. WT-generated vs Mut-generated (10 A) delta distributions
2. 10 A Mut-generated vs 6 A Mut-generated delta distributions
3. WT-generated vs 6 A Mut-generated

KS-test: sensitive to any difference in shape.
Mann-Whitney U: non-parametric comparison of medians.


In [ ]:
# -- Section 2: KS-test + Mann-Whitney --
from scipy import stats
import pandas as pd

wt_d   = gen[gen['source']=='WT_generated']['delta'].dropna()
mut_d  = gen[gen['source']=='Mut_generated']['delta'].dropna()
mut6_d = df6a['delta'].dropna() if not df6a.empty else pd.Series(dtype=float)

comparisons = [
    ('WT-gen (10A)',  'Mut-gen (10A)', wt_d,  mut_d),
    ('Mut-gen (10A)', 'Mut-gen (6A)',  mut_d, mut6_d),
    ('WT-gen (10A)',  'Mut-gen (6A)',  wt_d,  mut6_d),
]

print(f'{"Comparison":<35} {"KS stat":>8} {"KS p":>10} {"MW p":>10} {"Mean A":>8} {"Mean B":>8}')
print('-' * 90)
results = []
for label_a, label_b, a, b in comparisons:
    if len(b) < 2:
        print(f'{label_a} vs {label_b}: not enough data'); continue
    ks  = stats.ks_2samp(a, b)
    mw  = stats.mannwhitneyu(a, b, alternative='two-sided')
    sig = '*' if ks.pvalue < 0.05 else 'ns'
    print(f'{label_a+" vs "+label_b:<35} {ks.statistic:8.3f} {ks.pvalue:10.4f} {mw.pvalue:10.4f}'
          f' {a.mean():8.3f} {b.mean():8.3f}  {sig}')
    results.append({'comparison': f'{label_a} vs {label_b}',
                    'KS_stat': round(ks.statistic,3), 'KS_p': round(ks.pvalue,4),
                    'MW_p': round(mw.pvalue,4),
                    'mean_A': round(float(a.mean()),3), 'mean_B': round(float(b.mean()),3),
                    'significant': sig})

stats_df = pd.DataFrame(results)
stats_df.to_csv(f'{PROJECT_ROOT}/results/analysis/statistical_tests.csv', index=False)
print()
print('Interpretation (KS p-value):')
print('  p > 0.05 (ns)  -> distributions NOT significantly different')
print('  p < 0.05 (*)   -> distributions significantly different')


---
## Section 3 — Tanimoto Structural Selectivity Score

Vina delta measures *binding energy* selectivity.
Here we measure *structural* selectivity: how similar is each generated molecule
to known T790M-selective drugs (Osimertinib) vs WT-selective drugs (Erlotinib)?

**Structural selectivity score = Tanimoto(mol, Osimertinib) - Tanimoto(mol, Erlotinib)**

Positive: structurally closer to Osimertinib (T790M-biased scaffold)
Negative: structurally closer to Erlotinib (WT-biased scaffold)

We then plot this against the Vina delta to see if both signals agree.


In [ ]:
# -- Section 3: Tanimoto structural selectivity score --
from rdkit import Chem, DataStructs
from rdkit.Chem import AllChem
import pandas as pd, numpy as np, matplotlib.pyplot as plt

def morgan(smi, r=2, n=2048):
    mol = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(mol, r, n) if mol else None

# Reference fingerprints
osimertinib_smi = cfg['baselines']['osimertinib']['smiles']
erlotinib_smi   = cfg['baselines']['erlotinib']['smiles']
gefitinib_smi   = cfg['baselines']['gefitinib']['smiles']
fp_osi  = morgan(osimertinib_smi)
fp_erl  = morgan(erlotinib_smi)
fp_gef  = morgan(gefitinib_smi)

def struct_selectivity(smi):
    fp = morgan(smi)
    if fp is None or fp_osi is None or fp_erl is None:
        return None
    sim_osi = DataStructs.TanimotoSimilarity(fp, fp_osi)
    sim_erl = DataStructs.TanimotoSimilarity(fp, fp_erl)
    return sim_osi - sim_erl   # positive = T790M-biased

# Compute for all generated molecules
all_gen = pd.concat([gen, df6a], ignore_index=True) if not df6a.empty else gen.copy()
all_gen['struct_sel'] = all_gen['smiles'].apply(struct_selectivity)
all_gen = all_gen.dropna(subset=['struct_sel', 'delta'])

# Correlation: Vina delta vs structural selectivity
wt10  = all_gen[all_gen['source']=='WT_generated']
mu10  = all_gen[all_gen['source']=='Mut_generated']
mu6a  = all_gen[all_gen['source']=='Mut_6A']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: Vina delta vs structural selectivity
for sub, col, lbl in [(wt10,'#1f77b4','WT 10A'), (mu10,'#d62728','Mut 10A'), (mu6a,'#e67e22','Mut 6A')]:
    if len(sub):
        axes[0].scatter(sub['struct_sel'], sub['delta'],
                        c=col, alpha=0.55, s=40, label=lbl)

axes[0].axhline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].axvline(0, color='gray', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Structural selectivity score (Tanimoto vs Osimertinib minus Tanimoto vs Erlotinib)')
axes[0].set_ylabel('Vina delta = vina_Mut - vina_WT (kcal/mol)')
axes[0].set_title('Structural vs Binding Selectivity')
axes[0].legend(fontsize=9)

# Pearson correlation
from scipy.stats import pearsonr
r, p = pearsonr(all_gen['struct_sel'], all_gen['delta'])
axes[0].text(0.05, 0.95, f'r = {r:.3f}  p = {p:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             verticalalignment='top',
             bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

# Distribution of structural selectivity score by pool
for sub, col, lbl in [(wt10,'#1f77b4','WT 10A'), (mu10,'#d62728','Mut 10A')]:
    if len(sub):
        axes[1].hist(sub['struct_sel'], bins=15, alpha=0.55, color=col, label=lbl)

axes[1].axvline(0, color='gray', linewidth=0.8, linestyle='--', label='neutral')
axes[1].set_xlabel('Structural selectivity score')
axes[1].set_ylabel('Count')
axes[1].set_title('Structural Selectivity Distribution by Pool')
axes[1].legend()

plt.tight_layout()
out = f'{PROJECT_ROOT}/results/analysis/structural_selectivity.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
print('Structural selectivity score summary:')
for lbl, sub in [('WT 10A', wt10), ('Mut 10A', mu10), ('Mut 6A', mu6a)]:
    if len(sub):
        sc = sub['struct_sel']
        print(f'  {lbl:10s}: mean={sc.mean():.4f}  std={sc.std():.4f}  '
              f'T790M-biased(>0): {(sc>0).sum()}/{len(sc)}')

all_gen[['source','smiles','delta','struct_sel','vina_WT','vina_Mut']].to_csv(
    f'{PROJECT_ROOT}/results/analysis/selectivity_scores_combined.csv', index=False)
print('Saved ->', f'{PROJECT_ROOT}/results/analysis/selectivity_scores_combined.csv')


---
## Section 3.5 — Pharmacophore Filter

Post-hoc structural filter: which generated molecules contain key binding features
of known EGFR inhibitors and T790M-selective drugs?

| Feature | Biological role | Present in |
|---------|----------------|-----------|
| NH_hinge | H-bond donor to Met793 backbone | All EGFR drugs |
| Aromatic_N | H-bond acceptor from Met793 NH | All EGFR drugs |
| Bicyclic_arom | Hydrophobic packing (gatekeeper) | All EGFR drugs |
| Aminopyrimidine | 3rd-gen hinge motif (T790M drugs) | Osimertinib |
| Acrylamide | Covalent warhead to Cys797 | Osimertinib |

Molecules scoring **3+ out of 5** features are pharmacophore-confirmed hits.


In [ ]:
# -- Section 3.5: Pharmacophore filter --
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display
import pandas as pd

pharma_features = {
    'NH_hinge':        '[NH][c,n]',
    'Aromatic_N':      'n',
    'Bicyclic_arom':   'c1ccc2ccccc2c1',
    'Aminopyrimidine': '[NH]c1ncncc1',
    'Acrylamide':      'C=CC(=O)[NH]',
}
pats = {k: Chem.MolFromSmarts(v) for k, v in pharma_features.items()}

def score_pharma(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    hits = {k: bool(mol.HasSubstructMatch(v)) for k, v in pats.items()}
    hits['score'] = sum(hits.values())
    return hits

# Prevalence across all filtered molecules
print('=== Pharmacophore feature prevalence in filtered pools ===')
for pool_name, csv_path in [
    ('WT-generated (264)', f'{PROJECT_ROOT}/results/filtered/1M17_filtered.csv'),
    ('Mut-generated (284)', f'{PROJECT_ROOT}/results/filtered/4I22_filtered.csv'),
]:
    df_pool = pd.read_csv(csv_path)
    hl = [score_pharma(s) for s in df_pool['smiles']]
    hl = [h for h in hl if h is not None]
    n = len(hl)
    print(f'\n{pool_name}:')
    for feat in pharma_features:
        c = sum(h[feat] for h in hl)
        print(f'  {feat:22s}: {c:3d}/{n} ({100*c/n:.1f}%)')
    confirmed = sum(h['score'] >= 3 for h in hl)
    print(f'  {"pharma-confirmed(>=3)":22s}: {confirmed:3d}/{n} ({100*confirmed/n:.1f}%)')

# Analyse top-20 candidates
print('\n\n=== Top-20 pharmacophore breakdown ===')
top20 = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv')
records = []
for i, (_, row) in enumerate(top20.iterrows()):
    h = score_pharma(row['smiles'])
    if h:
        h.update({'rank': i+1, 'smiles': row['smiles'],
                  'composite': round(row['composite'], 3),
                  'vina_Mut': row['vina_Mut'],
                  'delta': row['delta'], 'QED': row['QED']})
        records.append(h)

pharma_top = pd.DataFrame(records)
print(pharma_top[['rank', 'vina_Mut', 'delta', 'QED', 'score',
                   'NH_hinge', 'Aromatic_N', 'Bicyclic_arom',
                   'Aminopyrimidine', 'Acrylamide']].to_string(index=False))
pharma_top.to_csv(f'{PROJECT_ROOT}/results/analysis/pharmacophore_top20.csv', index=False)

# Show confirmed candidates as structure grid
confirmed = pharma_top[pharma_top['score'] >= 3].head(8)
print(f'\nTop-20 molecules with >=3 pharmacophore features: {len(confirmed)}')
if len(confirmed):
    mols = [Chem.MolFromSmiles(s) for s in confirmed['smiles']]
    legs = []
    for _, row in confirmed.iterrows():
        legs.append(
            'Rank #' + str(int(row['rank'])) +
            '  pharma=' + str(int(row['score'])) +
            '\nvina_Mut=' + str(round(row['vina_Mut'], 1)) +
            '  QED=' + str(round(row['QED'], 2))
        )
    grid = Draw.MolsToGridImage(mols, molsPerRow=4, subImgSize=(280, 220),
                                 legends=legs)
    display(grid)
    try:
        grid.save(f'{PROJECT_ROOT}/results/analysis/pharmacophore_confirmed.png')
    except AttributeError:
        with open(f'{PROJECT_ROOT}/results/analysis/pharmacophore_confirmed.png', 'wb') as f:
            f.write(grid.data)
    print('Saved -> results/analysis/pharmacophore_confirmed.png')


---
## Section 4 — Top Candidates Composite Ranking

Composite score combining:
- **Binding strength** on T790M: -vina_Mut (normalized)
- **Drug-likeness**: QED
- **Synthesizability**: 1 - SA/10
- **Novelty**: 1 - MaxTanimotoToBaseline

Score = 0.40 * binding + 0.30 * QED + 0.20 * synth + 0.10 * novelty

Only Mut-generated (10Å) molecules with QED and SA data are ranked.


In [ ]:
# -- Section 4: Composite ranking + Top-10 --
import pandas as pd, numpy as np
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

# Merge vina scores with properties from filtered CSVs
mut_scored = gen[gen['source']=='Mut_generated'][['smiles','vina_WT','vina_Mut','delta']].copy()
wt_scored  = gen[gen['source']=='WT_generated'][['smiles','vina_WT','vina_Mut','delta']].copy()

props_mut = filt_mut[['smiles','QED','SA','MaxTanimotoToBaseline','MW','LogP','TPSA']].copy()
props_wt  = filt_wt[['smiles','QED','SA','MaxTanimotoToBaseline','MW','LogP','TPSA']].copy()

combined_props = pd.concat([props_mut, props_wt], ignore_index=True).drop_duplicates('smiles')
all_scored = pd.concat([mut_scored, wt_scored], ignore_index=True)
merged = all_scored.merge(combined_props, on='smiles', how='inner')
print(f'Molecules with both vina + properties: {len(merged)}')

# Normalize each component to [0,1]
def minmax(s):
    lo, hi = s.min(), s.max()
    return (s - lo) / (hi - lo) if hi > lo else pd.Series(0.5, index=s.index)

merged['score_binding']  = minmax(-merged['vina_Mut'])
merged['score_qed']      = minmax(merged['QED'])
merged['score_synth']    = minmax(1 - merged['SA'] / 10)
merged['score_novelty']  = minmax(1 - merged['MaxTanimotoToBaseline'])

merged['composite'] = (0.40 * merged['score_binding'] +
                        0.30 * merged['score_qed'] +
                        0.20 * merged['score_synth'] +
                        0.10 * merged['score_novelty'])

top20 = merged.nlargest(20, 'composite').reset_index(drop=True)
top20.to_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv', index=False)

cols = ['smiles','vina_WT','vina_Mut','delta','QED','SA','MW','LogP','composite']
print(top20[cols].to_string(index=True))


In [ ]:
# -- Top-10 structure grid --
from rdkit.Chem import Draw
from IPython.display import display
import pandas as pd

top20 = pd.read_csv(f'{PROJECT_ROOT}/results/analysis/top20_candidates.csv')
top10 = top20.head(10)

mols = [Chem.MolFromSmiles(s) for s in top10['smiles']]
legends = [
    f'#{i+1}  score={row["composite"]:.3f}\n'
    f'vina_Mut={row["vina_Mut"]:.1f}  delta={row["delta"]:+.2f}\n'
    f'QED={row["QED"]:.2f}  SA={row["SA"]:.1f}'
    for i, (_, row) in enumerate(top10.iterrows())
]
img = Draw.MolsToGridImage(mols, molsPerRow=5, subImgSize=(300, 240), legends=legends)
display(img)
# Save
img.save(f'{PROJECT_ROOT}/results/analysis/top10_structure_grid.png')
print('Saved -> results/analysis/top10_structure_grid.png')


---
## Section 5 — 3D Case Study: Best T790M-selective Candidate

We take the molecule with the strongest T790M binding (most negative vina_Mut)
from the Mut-generated pool, generate its 3D conformer, and visualize it
alongside the T790M (4I22) binding pocket.

The T790M gatekeeper residue (Met790) is highlighted to show how the
molecule occupies the ATP-binding site near the mutation.


In [ ]:
# -- Section 5: 3D visualization with py3Dmol --
import py3Dmol
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from IPython.display import display
import os

# Find best T790M-binding molecule from Mut pool
mut_only = gen[gen['source']=='Mut_generated'].copy()
best_row = mut_only.loc[mut_only['vina_Mut'].idxmin()]
best_smi = best_row['smiles']
print(f'Best candidate: vina_Mut={best_row["vina_Mut"]:.2f}  '
      f'vina_WT={best_row["vina_WT"]:.2f}  delta={best_row["delta"]:+.3f}')
print(f'SMILES: {best_smi}')

# 1) Show 2D structure
mol2d = Chem.MolFromSmiles(best_smi)
img2d = Draw.MolToImage(mol2d, size=(350, 260))
display(img2d)

# 2) Generate 3D conformer
mol3d = Chem.AddHs(Chem.MolFromSmiles(best_smi))
result = AllChem.EmbedMolecule(mol3d, AllChem.ETKDGv3())
if result < 0:
    print('3D embedding failed, trying ETKDG v2...')
    result = AllChem.EmbedMolecule(mol3d, AllChem.ETKDGv2())
AllChem.MMFFOptimizeMolecule(mol3d)

# Save as SDF for py3Dmol
lig_sdf = '/tmp/best_candidate.sdf'
w = Chem.SDWriter(lig_sdf)
w.write(mol3d); w.close()

# 3) Load pocket PDB
pocket_pdb = f'{PROJECT_ROOT}/data/pockets/4I22_pocket.pdb'
with open(pocket_pdb) as f:
    pocket_str = f.read()
with open(lig_sdf) as f:
    lig_str = f.read()

# 4) py3Dmol visualization
view = py3Dmol.view(width=720, height=500)

# Protein pocket as cartoon + surface
view.addModel(pocket_str, 'pdb')
view.setStyle({'model': 0},
              {'cartoon': {'color': 'spectrum', 'opacity': 0.6},
               'stick': {'radius': 0.1, 'color': 'gray', 'opacity': 0.3}})

# Highlight Met790 (T790M gatekeeper)
view.addStyle({'model': 0, 'resi': '790'},
              {'stick': {'color': '#e74c3c', 'radius': 0.35},
               'sphere': {'color': '#e74c3c', 'radius': 0.5, 'opacity': 0.6}})
view.addLabel('Met790 (T790M)', {'model': 0, 'resi': '790'},
              {'backgroundColor': '#e74c3c', 'fontColor': 'white',
               'fontSize': 12, 'fontOpacity': 0.9})

# Ligand as ball-and-stick
view.addModel(lig_str, 'sdf')
view.setStyle({'model': 1},
              {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.25},
               'sphere': {'colorscheme': 'cyanCarbon', 'radius': 0.18}})

view.zoomTo({'model': 1})
view.setBackgroundColor('white')
view.show()
print('Note: ligand shown as computed 3D conformer (not actual docked pose);')
print('it illustrates the molecule size and geometry relative to the T790M pocket.')


---
## Section 6 — Findings Summary


In [ ]:
# -- Section 6: Summary table --
import pandas as pd

summary = pd.DataFrame([
    {'Finding': 'Generation quality',
     'Result': '1987 unique molecules, 100% valid, 26-28% drug-like',
     'Evidence': 'nb04/06/05 generation + filtering'},

    {'Finding': 'Pool structural diversity',
     'Result': '0.8% SMILES overlap, mean Tanimoto 0.285',
     'Evidence': 'Cross-pool Tanimoto analysis (nb05)'},

    {'Finding': 'RQ1: TargetDiff generates drug-like molecules',
     'Result': 'YES - validated by QED/SA/Lipinski/PAINS filter',
     'Evidence': 'Filter funnel: 26-28% pass all criteria'},

    {'Finding': 'RQ2: Pocket-conditioning produces selectivity',
     'Result': 'NO - both pools cluster on equal-binding diagonal',
     'Evidence': 'KS-test p > 0.05; MW p > 0.05 (nb07 + Section 2)'},

    {'Finding': 'RQ3: Pocket radius (Exp A)',
     'Result': '6A generates fragments (MW<200), no selectivity gain',
     'Evidence': 'Experiment A: 6A median delta +0.2 vs 10A -0.15 (nb08)'},

    {'Finding': 'Structural selectivity (Exp B-C)',
     'Result': 'Corr(Tanimoto_sel, Vina_delta) r ~ 0.0-0.1',
     'Evidence': 'Structural selectivity score (Section 3 above)'},

    {'Finding': 'Best T790M-selective candidate',
     'Result': 'vina_Mut = -11.4, QED=XX, SA=XX (from top20_candidates.csv)',
     'Evidence': 'Composite ranking (Section 4)'},

    {'Finding': 'Root cause of selectivity failure',
     'Result': 'Model training on CrossDocked2020 treats each pocket independently; '
               'never learned mutation-pair relationships',
     'Evidence': 'Literature: TargetDiff ICLR2023 training objective'},
])

print(summary.to_string(index=False))
summary.to_csv(f'{PROJECT_ROOT}/results/analysis/findings_summary.csv', index=False)
print()
print('All figures saved to results/analysis/')
import glob, os
for f in sorted(glob.glob(f'{PROJECT_ROOT}/results/analysis/*.png') +
                glob.glob(f'{PROJECT_ROOT}/results/analysis/*.csv')):
    print(f'  {os.path.basename(f)}  ({os.path.getsize(f)/1024:.1f} KB)')
